# Reproduce the GCL headline stats (full scale, n=30 seeds)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jstiltner/gcl/blob/main/notebooks/reproduce_headline_stats.ipynb)

Runs the real agent-based simulations behind two flagship claims on
[jasonstiltner.com](https://jasonstiltner.com/projects/grounded-commitment-learning):

- **Punishment Paradox** — does increasing consequences for commitment violations *decrease*
  cooperation? (`16_consequence_severity_sweep.py`)
- **Hart-Moore incomplete contract theory** — does GCL's failure-first specification reduce
  hold-up incidents versus an incomplete-contract baseline? (`21_incomplete_contract_theory.py`)

This is the **full published scale** (n=30 seeds per condition). The CI badge on the repo's
README runs a reduced n=5-seed version on every push — same code path
(`experiments/derive_real_headline_stats.py`), different `--ci` flag, different (wider) confidence
intervals by design. See `CHANGELOG.md` for why this script exists: the numbers previously
published for these two claims traced to a synthetic-data generator, not these simulations —
this notebook and its CI counterpart are what actually back the published numbers now.

In [ ]:
!git clone --depth 1 https://github.com/jstiltner/gcl.git
%cd gcl
!pip install -q numpy scipy matplotlib

In [ ]:
# Full scale: n=30 seeds, matching the seed count originally claimed for both experiments.
# Takes about a minute on a Colab CPU runtime.
!python experiments/derive_real_headline_stats.py

In [ ]:
import json

with open("results/real_headline_stats/real_headline_stats_full.json") as f:
    data = json.load(f)

pp = data["punishment_paradox"]
hm = data["hart_moore"]

print("Punishment Paradox")
print(f"  r = {pp['correlation']['r']:.3f}, p = {pp['correlation']['p']:.2e}")
print(f"  no vs full consequences: t = {pp['no_vs_full_consequences']['t']:.2f}, "
      f"d = {pp['no_vs_full_consequences']['d']:.2f}")
print()
print("Hart-Moore")
print(f"  hold-up reduction: {hm['holdup_reduction_pct']:.1f}%")
for name, t in hm["tests"].items():
    print(f"  {name}: t = {t['t']:.2f}, d = {t['d']:.2f}")

In [ ]:
# Plot the real Punishment Paradox curve (matches jasonstiltner.com's chart)
import matplotlib.pyplot as plt

levels = sorted(float(k) for k in pp["cooperation_by_level"].keys())
means = [pp["cooperation_by_level"][str(l)]["mean"] for l in levels]
stds = [pp["cooperation_by_level"][str(l)]["std"] for l in levels]

plt.errorbar(levels, means, yerr=stds, marker="o", capsize=4)
plt.xlabel("Consequence severity")
plt.ylabel("Cooperation rate")
plt.title(f"Punishment Paradox (real simulation, n=30 seeds) — r={pp['correlation']['r']:.3f}")
plt.ylim(0, 0.6)
plt.grid(alpha=0.3)
plt.show()